In [17]:
import numpy as np
import yfinance as yf
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import RandomizedSearchCV

In [18]:
df = yf.download("RELIANCE.NS", start="2011-01-01", end="2024-01-01",auto_adjust=True)
df.head()
reliance = yf.Ticker("RELIANCE.NS")
df = df.droplevel("Ticker", axis=1)
train = df[:"2020-12-31"]  # Jan 2011 – Dec 2020 (10 years)
test  = df["2021-01-01":]  # Jan 2021 – Jan 2024  (3 years)

[*********************100%***********************]  1 of 1 completed


In [19]:
def create_target(data, horizon=5):
    # Binary target: 1 if Close(T+horizon) > Close(T), else 0.
    if isinstance(data.columns, pd.MultiIndex):
        data = data.droplevel("Ticker", axis=1)
    close = data["Close"]
    future_close = close.shift(-horizon)
    target = (future_close > close).astype(int)
    target.name = "target"
    return target


In [20]:
def build_features(data):
    """
    Engineer features from OHLCV data for logistic regression.
    Returns a DataFrame of features aligned with the original index.
    """
    # Flatten MultiIndex columns if present (yfinance returns multi-level cols)
    if isinstance(data.columns, pd.MultiIndex):
        data = data.droplevel("Ticker", axis=1)

    close = data["Close"]
    volume = data["Volume"]

    feat = pd.DataFrame(index=data.index)

    # Price returns
    feat["ret_1"]  = close.pct_change(1)    # 1-day return
    feat["ret_5"]  = close.pct_change(5)    # 5-day return
    feat["ret_10"] = close.pct_change(10)   # 10-day return
    feat["ret_20"] = close.pct_change(20)   # 20-day return

    # Moving averages (ratio to current price)
    feat["ma5_ratio"]  = close / close.rolling(5).mean()
    feat["ma10_ratio"] = close / close.rolling(10).mean()
    feat["ma20_ratio"] = close / close.rolling(20).mean()
    feat["ma50_ratio"] = close / close.rolling(50).mean()

    # Volatility (rolling std of returns)
    feat["vol_5"]  = close.pct_change().rolling(5).std()
    feat["vol_20"] = close.pct_change().rolling(20).std()

    # RSI (14-day)
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss
    feat["rsi_14"] = 100 - (100 / (1 + rs))

    # Volume change
    feat["vol_change_1"] = volume.pct_change(1)
    feat["vol_change_5"] = volume.pct_change(5)

    # Rolling Z-scores (how far price is from recent average, in std devs)
    feat["zscore_20"] = (close - close.rolling(20).mean()) / close.rolling(20).std()
    feat["zscore_50"] = (close - close.rolling(50).mean()) / close.rolling(50).std()

    # Replace any inf values with NaN (e.g. from zero-volume days)
    feat.replace([np.inf, -np.inf], np.nan, inplace=True)

    return feat

In [22]:
def run_random_forest(train_df, test_df, horizon=5):
    # Build features & target
    X_train = build_features(train_df)
    y_train = create_target(train_df, horizon)
    X_test  = build_features(test_df)
    y_test  = create_target(test_df, horizon)

    # Drop NaN rows
    train_combined = pd.concat([X_train, y_train], axis=1).dropna()
    test_combined  = pd.concat([X_test, y_test], axis=1).dropna()

    X_train_clean = train_combined.drop("target", axis=1)
    y_train_clean = train_combined["target"]
    X_test_clean  = test_combined.drop("target", axis=1)
    y_test_clean  = test_combined["target"]

    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_clean)
    X_test_scaled  = scaler.transform(X_test_clean)

    # Train Random Forest
    model = RandomForestClassifier(
        n_estimators=500,     # number of trees
        max_depth=5,         # limit depth to prevent overfitting
        min_samples_split=20, # minimum samples to split a node
        min_samples_leaf=10,  # minimum samples in a leaf
        max_features="sqrt",  # use sqrt(n_features) at each split
        random_state=42,
        n_jobs=-1             # use all CPU cores
    )
    model.fit(X_train_scaled, y_train_clean)

    # Predict
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred  = model.predict(X_test_scaled)

    # Results
    print("\n" + "=" * 60)
    print("  RANDOM FOREST – Predict Close(T+5) > Close(T)")
    print("=" * 60)

    print(f"\n--- Train Set Results ---")
    print(f"Accuracy: {accuracy_score(y_train_clean, y_train_pred):.4f}")
    print(classification_report(y_train_clean, y_train_pred, target_names=["Down/Flat", "Up"]))

    print(f"--- Test Set Results ---")
    print(f"Accuracy: {accuracy_score(y_test_clean, y_test_pred):.4f}")
    print(classification_report(y_test_clean, y_test_pred, target_names=["Down/Flat", "Up"]))

    print("Confusion Matrix (Test):")
    print(confusion_matrix(y_test_clean, y_test_pred))

    # Feature importance
    feature_names = X_train_clean.columns
    importances = pd.Series(model.feature_importances_, index=feature_names)
    importances = importances.sort_values(ascending=False)
    print("\nFeature Importances:")
    print(importances.to_string())

    return model, scaler

In [25]:
param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 10, 15, None],
    "min_samples_split": [5, 10, 20, 30, 50],
    "min_samples_leaf": [5, 10, 15, 20, 30],
    "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7]
}
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist,
    n_iter=50,          # try 50 random combinations
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    random_state=42,
    verbose=1
)
random_search.fit(X_train_scaled, y_train_clean)
print(f"Best params: {random_search.best_params_}")
print(f"Best CV accuracy: {random_search.best_score_:.4f}")

NameError: name 'X_train_scaled' is not defined

In [24]:
model, scaler = run_random_forest(train, test, horizon=5)


  RANDOM FOREST – Predict Close(T+5) > Close(T)

--- Train Set Results ---
Accuracy: 0.6826
              precision    recall  f1-score   support

   Down/Flat       0.76      0.46      0.58      1120
          Up       0.65      0.87      0.75      1290

    accuracy                           0.68      2410
   macro avg       0.71      0.67      0.66      2410
weighted avg       0.70      0.68      0.67      2410

--- Test Set Results ---
Accuracy: 0.5592
              precision    recall  f1-score   support

   Down/Flat       0.54      0.36      0.44       324
          Up       0.57      0.73      0.64       368

    accuracy                           0.56       692
   macro avg       0.56      0.55      0.54       692
weighted avg       0.56      0.56      0.54       692

Confusion Matrix (Test):
[[118 206]
 [ 99 269]]

Feature Importances:
vol_20          0.115606
ma50_ratio      0.100585
vol_5           0.094236
ret_20          0.083652
ret_10          0.083103
zscore_20       